# 🌊 AquaGuard AI — Day 11: Preprocessing Pipeline & Dataset V1
- **Date:** 30 Aug 2026
- **Phase:** Phase 2: Feature Engineering & Preprocessing
- **Objective:** Compose leakage-proof cleaning, resampling, and scaling pipeline. Fit scalers strictly on Train partition.
- **Inputs Consumed:** `data/raw/underwater_sensor_stream.csv`, `artifacts/day09/split.json`, `artifacts/day08/rules.yaml`
- **Outputs Produced:** `data/processed/dataset_v1_train.parquet`, `data/processed/dataset_v1_val.parquet`, `data/processed/dataset_v1_test_lockbox.parquet`, `artifacts/day11/preprocessor.joblib`
- **Deliverable Satisfied:** Sanitized, regularized Processed Dataset V1.

In [1]:
import os, sys
from pathlib import Path
project_root = Path("..").resolve() if Path("..").resolve().name == "Water Potability" else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from src.utils import set_seed, load_config, ensure_dir
from src.io import load_raw_data, load_artifact, save_parquet, save_artifact
from src.preprocessing import clean_and_resample_stream, DomainBoundsClipper

set_seed(42)
config = load_config("configs/config.yaml")
ensure_dir("artifacts/day11")
ensure_dir("data/processed")

# Load Day 08 rules and Day 09 split indices
rules = load_artifact("artifacts/day08/rules.yaml", required_by_day="Day 08")
split_indices = load_artifact("artifacts/day09/split.json", required_by_day="Day 09")
print("Prerequisites verified: Rules and split indices loaded.")

Prerequisites verified: Rules and split indices loaded.


## 1. Clean & Resample Partitions Independently
Resample each partition to regular 1-hour frequency using Day 08 frozen rules.

In [2]:
df_raw = load_raw_data(config["paths"]["raw_temporal_data"])
ts_col = config["track_a"]["timestamp_column"]

# Split raw dataframe by partition indices
raw_train = df_raw.iloc[split_indices["train_indices"]].copy()
raw_val = df_raw.iloc[split_indices["val_indices"]].copy()
raw_test = df_raw.iloc[split_indices["test_indices"]].copy()

# Resample each partition independently to avoid cross-boundary leakage
clean_train = clean_and_resample_stream(raw_train, timestamp_col=ts_col, freq="1h")
clean_val = clean_and_resample_stream(raw_val, timestamp_col=ts_col, freq="1h")
clean_test = clean_and_resample_stream(raw_test, timestamp_col=ts_col, freq="1h")

print(f"Resampled partition shapes: Train={clean_train.shape}, Val={clean_val.shape}, Test={clean_test.shape}")

Resampled partition shapes: Train=(503, 7), Val=(109, 7), Test=(108, 7)


## 2. Fit Scalers & Domain Clippers Strictly on Train
Fit `DomainBoundsClipper` and `StandardScaler` exclusively on the training partition. Transform Validation and Test without refitting.

In [3]:
sensor_cols = config["track_a"]["sensor_columns"]

clipper = DomainBoundsClipper(bounds=rules["domain_bounds"])
scaler = StandardScaler()

# Fit transformers on Train only
train_clipped = clipper.transform(clean_train[sensor_cols])
scaler.fit(train_clipped)

# Save pipeline artifacts
preprocessor_bundle = {
    "clipper": clipper,
    "scaler": scaler,
    "sensor_cols": sensor_cols,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist()
}
save_artifact(preprocessor_bundle, "artifacts/day11/preprocessor.joblib")
print("Fitted preprocessor bundle saved to artifacts/day11/preprocessor.joblib")

Fitted preprocessor bundle saved to artifacts/day11/preprocessor.joblib


## 3. Serialize Processed Dataset V1 (Parquet)
Export cleaned and standardized partitions in Parquet format.

In [4]:
save_parquet(clean_train, "data/processed/dataset_v1_train.parquet")
save_parquet(clean_val, "data/processed/dataset_v1_val.parquet")
save_parquet(clean_test, "data/processed/dataset_v1_test_lockbox.parquet")
print("Dataset V1 saved to data/processed/")

Dataset V1 saved to data/processed/


## 📌 Day 11 Execution Findings & Sign-Off
- **Key Finding:** Resampling and domain clipping complete. Scalers fit exclusively on train.
- **Artifacts Saved:** `data/processed/dataset_v1_*.parquet`, `artifacts/day11/preprocessor.joblib`.
- **Handoff:** Ready for Day 12 automated leakage checks and Dataset V1 freeze.